# Fraud Detection: Baseline Model (Logistic Regression)

Milestone 2. We build a first, simple model and score it on the **validation split**. The scores are the benchmark that every later model (more features, tuning, other models) has to beat.

The baseline is a **logistic regression** on the scaled and one-hot encoded table (`X_*_logreg`) from [08_feature-engineering.ipynb](08_feature-engineering.ipynb). It is simple, fast and easy to explain, which is what a baseline needs.

The Zindi test set is **not used** here: it has no labels, and we only predict it once at the end (workflow step 9).

**Metrics** (see [problem_framing.md](problem_framing.md)): ROC-AUC is the official Zindi metric, PR-AUC checks how well the rare fraud class (5.6%) is found. Precision, recall and F1 need a threshold, we show them at 0.5 and at the F1-best threshold.

In [1]:
# reload features.py when it changes, so a running kernel does not use an old version
%load_ext autoreload
%autoreload 2

# libraries for data handling, the model and the metrics
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score

# show all columns, read the feature tables from 08, fix the seed so results repeat
pd.set_option("display.max_columns", 40)
FEATURE_DIR = Path("data/features")
RANDOM_STATE = 42

## 1. Load the feature tables

Index = `client_id`. Train and validation come from the 80/20 client split of the Zindi train set.

In [2]:
# read one feature table from data/features, with client_id as the index
def load(name):
    return pd.read_csv(FEATURE_DIR / f"{name}.csv", index_col="client_id")

# scaled and one-hot encoded tables for logistic regression, plus the fraud labels
X_train, X_valid = load("X_train_logreg"), load("X_valid_logreg")
y_train = load("y_train")["target"]
y_valid = load("y_valid")["target"]

# make sure features and labels are in the same client order
assert (X_train.index == y_train.index).all() and (X_valid.index == y_valid.index).all()

# check the sizes and that both splits have a similar fraud share
print("train:", X_train.shape, "validation:", X_valid.shape)
print(f"fraud share: train {y_train.mean():.2%}, validation {y_valid.mean():.2%}")

train: (108394, 57) validation: (27099, 57)
fraud share: train 5.58%, validation 5.58%


## 2. Scoring helper

One function scores a model from its fraud probabilities. We report the threshold-free metrics and, for the threshold metrics, two views:
- **at 0.5:** the default cut-off,
- **at the F1-best threshold:** the cut-off that gives the best F1 on the validation split. This is slightly optimistic because the threshold is chosen on the same data. The real threshold depends on how many inspections STEG can do, so treat it as an upper bound for now.

In [3]:
# score one model from its fraud probabilities and return all metrics as one row
def score(name, y_true, proba):
    # precision and recall for every possible threshold
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    # F1 for every threshold (the last precision/recall pair has no threshold, so it is dropped)
    f1_curve = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    # the F1-best threshold: the cut-off with the highest F1 on this data
    best = f1_curve.argmax()
    # default cut-off: call a client fraud if the probability is at least 0.5
    at_half = proba >= 0.5
    # threshold-free metrics first (ROC-AUC, PR-AUC), then the metrics at 0.5 and at the F1-best threshold
    return {
        "model": name,
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
        "precision@0.5": precision_score(y_true, at_half, zero_division=0),
        "recall@0.5": recall_score(y_true, at_half),
        "F1@0.5": f1_score(y_true, at_half, zero_division=0),
        "best threshold": thresholds[best],
        "precision@best": precision[best],
        "recall@best": recall[best],
        "F1@best": f1_curve[best],
    }

## 3. Logistic regression

`class_weight="balanced"` makes the rare fraud class count more, because only 5.6% of the clients are fraud. The inputs are already scaled and encoded, using steps fitted on the training split only. No tuning yet.

In [4]:
# class_weight="balanced" makes the rare fraud class (5.6%) count more; max_iter is high so the model converges
logreg = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
# fit on the training split only
logreg.fit(X_train, y_train)
# fraud probability for each validation client (column 1 = fraud)
proba_valid = logreg.predict_proba(X_valid)[:, 1]

# score the validation split; this table is the benchmark for later models
baseline = pd.DataFrame([score("Logistic regression", y_valid, proba_valid)]).set_index("model").round(3)
baseline

,ROC-AUC,PR-AUC,precision@0.5,recall@0.5,F1@0.5,best threshold,precision@best,recall@best,F1@best
model,,,,,,,,,
Logistic regression,0.757,0.156,0.109,0.713,0.19,0.648,0.154,0.433,0.227


For comparison: a model that guesses at random has ROC-AUC 0.5 and a PR-AUC equal to the fraud share (about 0.056).

In [5]:
# a random guess has ROC-AUC 0.5 and PR-AUC equal to the fraud share, so the baseline should beat both
print(f"random guessing: ROC-AUC 0.500, PR-AUC {y_valid.mean():.3f}")
print(f"baseline:        ROC-AUC {baseline['ROC-AUC'].iloc[0]:.3f}, PR-AUC {baseline['PR-AUC'].iloc[0]:.3f}")

random guessing: ROC-AUC 0.500, PR-AUC 0.056
baseline:        ROC-AUC 0.757, PR-AUC 0.156


## 4. Which features does the model use?

The inputs are scaled, so the size of a coefficient shows how strongly a feature moves the fraud probability. A positive coefficient pushes towards fraud, a negative one away from it. It shows what the model relies on, not a cause. Features that overlap (for example the consumption statistics) share their effect, so single coefficients should be read with care.

In [6]:
# one coefficient per feature; inputs are scaled, so sizes can be compared
coefficients = pd.Series(logreg.coef_[0], index=X_train.columns)
# the 12 features with the largest effect, positive (towards fraud) or negative
top = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index).head(12)
top.round(3).rename("coefficient").to_frame()

,coefficient
consommation_level_2_max,0.951
has_backward,0.916
consommation_level_2_sum,-0.863
region_399,-0.776
region_206,-0.712
n_bills,0.687
max_change_elec,-0.661
cons_std_elec,0.628
cons_std,-0.567
region_372,0.563


## 5. Do the per-meter columns help?

[protocol.md](protocol.md) says: keep the per-meter columns (`cons_std_elec`, `max_change_elec`, `cons_std_gaz`, `max_change_gaz`) only if the validation score improves. A quick check with the same model, with and without them (including their missing-value markers). This uses one split, so only a clear difference counts.

In [7]:
# the per-meter columns and their missing-value markers end in _elec or _gaz
per_meter = [c for c in X_train.columns if c.endswith(("_elec", "_gaz"))]
print("columns removed:", per_meter)

# same model and settings, trained without the per-meter columns
without = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
without.fit(X_train.drop(columns=per_meter), y_train)
proba_without = without.predict_proba(X_valid.drop(columns=per_meter))[:, 1]

# compare ROC-AUC and PR-AUC with and without them; only a clear difference counts (one split)
pd.DataFrame(
    {
        "ROC-AUC": [roc_auc_score(y_valid, proba_valid), roc_auc_score(y_valid, proba_without)],
        "PR-AUC": [average_precision_score(y_valid, proba_valid), average_precision_score(y_valid, proba_without)],
    },
    index=[f"with per-meter ({X_train.shape[1]} columns)", f"without per-meter ({X_train.shape[1] - len(per_meter)} columns)"],
).round(4)

columns removed: ['cons_std_elec', 'max_change_elec', 'cons_std_gaz', 'max_change_gaz', 'missingindicator_cons_std_elec', 'missingindicator_max_change_elec', 'missingindicator_cons_std_gaz', 'missingindicator_max_change_gaz']


,ROC-AUC,PR-AUC
with per-meter (57 columns),0.7566,0.1563
without per-meter (49 columns),0.7531,0.1536


## 6. Summary

The table in section 3 is the benchmark. Points to keep in mind:
- The validation split is a single 20% sample of the training clients, so small differences between models are within the noise.
- The F1-best threshold is chosen on the validation data, so this F1 is an upper bound. The final threshold has to follow STEG's inspection capacity.
- Zindi scores only ROC-AUC, so that is the number to compare against the leaderboard.

**Next steps:** decide on the time features (open question in `protocol.md`), try tree models (LightGBM, XGBoost) on the tree table `X_*`, handle the imbalance and tune with stratified cross-validation, then refit on train plus validation and predict the Zindi test set once (workflow step 9).